# SimpleElasticSolid Demo

In [ ]:
import MeshFEM, mesh, mesh_energy, py_newton_optimizer, benchmark
import simple_elastic_solid, elastic_solid

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/bunny_coarse.msh')
x = mesh_energy.NodalVars(m, 3)

In [ ]:
es = simple_elastic_solid.SimpleElasticSolid(m.vertices(), m.elements(), x)
p = py_newton_optimizer.NewtonMultiobjectiveProblem(x, [es])

In [ ]:
import fd_validation
fd_validation.gradHessConvergencePlot(p)

In [ ]:
# Visualization
import viewer
em = MeshFEM.EmbeddedMesh(m, x)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
# Configure boundary conditions: glue to ground, pull ear up.
import sim_utils
groundVars = sim_utils.getBBoxVars(m, sim_utils.BBoxFace.MIN_Y, tol=4)
pullVars = sim_utils.getBBoxVars(m, sim_utils.BBoxFace.MAX_Y, tol=0.001)

xval = x.getVars()
xval[pullVars[1]] += 20
x.setVars(xval)
v.update()

p.setFixedVars(groundVars + pullVars)

In [ ]:
p.setCustomIterationCallback(v.updater())
benchmark.reset()
p.optimizer().optimize();
benchmark.report()